# 11 — Intervencija u faktorskom prostoru (HRP/HERC/NCO)

Faza 3: **F3.2** (walk-forward faktorskih verzija), **F3.3** (parne razlike
korelacijska↔faktorska — test H2, dio nagiba), **F3.4** (stabilnost klastera
ARI + obrtaj — test H2, dio 2) i **F3.5** (odluka o K: primarno + robusnost).

Jedina manipulirana varijabla naspram Faze 1 je **ulazno stablo**: Wardovo
klasteriranje na standardiziranim FF5 značajkama (`factor_cluster`) umjesto
korelacijskog stabla. Univerzum, Ledoit–Wolf Σ i `w_max` identični su
korelacijskoj grani (`run_hierarchical_walk_forward(tree_space="factor")`),
pa razliku u težinama nosi isključivo hijerarhija. Veza je Wardova u oba
kraka usporedbe (korekcija K1), tako da Faza 3 mijenja samo prostor.

In [1]:
%load_ext autoreload
%autoreload 2

import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import fcluster
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s")

from src.backtest import (
    HIERARCHICAL_FACTOR_PORTFOLIO_NAMES,
    generate_rolling_windows,
    run_hierarchical_walk_forward,
)
from src.clustering import (
    FACTOR_FEATURE_COLUMNS,
    ari_between_consecutive_windows,
    factor_cluster,
)
from src.hierarchical import build_correlation_tree, nco_weights
from src.portfolio import ledoit_wolf_cov
from src.evaluation import (
    FACTOR_COLUMNS,
    annualized_vol,
    apply_costs,
    block_bootstrap_diff,
    max_drawdown,
    sharpe_ratio,
    style_concentration,
    turnover_per_window,
)
from src.utils import (
    BLOCK_SIZE_MONTHS,
    N_BOOTSTRAP_RETURNS,
    PRIMARY_K,
    PROCESSED_DATA_DIR,
    RANDOM_SEED,
    TABLES_DIR,
    TC_BPS,
    W_MAX,
    ensure_dirs,
    set_seed,
)

ensure_dirs()
set_seed()

In [2]:
monthly_returns = pd.read_csv(
    PROCESSED_DATA_DIR / "monthly_returns.csv", index_col=0, parse_dates=True
)
factor_exposures = pd.read_csv(PROCESSED_DATA_DIR / "factor_exposures.csv")
factor_clusters = pd.read_csv(PROCESSED_DATA_DIR / "factor_clusters.csv")
correlation_clusters = pd.read_csv(PROCESSED_DATA_DIR / "correlation_clusters.csv")
metadata = pd.read_csv(PROCESSED_DATA_DIR / "metadata.csv")
membership = pd.read_csv(
    PROCESSED_DATA_DIR / "membership.csv",
    parse_dates=["start_date", "end_date"],
)
factors = pd.read_csv(
    PROCESSED_DATA_DIR / "factors.csv", index_col=0, parse_dates=True
)

windows = generate_rolling_windows()
K = PRIMARY_K  # primarni K za HERC/NCO (config.primary_k); HRP je K-neovisan
print("Prozori:", len(windows), "|", windows[0].label, "->", windows[-1].label, "| K =", K)

Prozori: 13 | 2012-12 -> 2024-12 | K = 10


## 11.1 F3.2 — Unaprijedni hod faktorskih verzija

`run_hierarchical_walk_forward(tree_space="factor")` po prozoru dohvaća šest
FF5 značajki (`FACTOR_FEATURE_COLUMNS`) iz `factor_exposures`, gradi jednu
Wardovu hijerarhiju (`factor_cluster`) i pušta `hrp_factor`, `herc_factor` i
`nco_factor` kroz isti klizni prozor kao Faza 1. Paneli se spremaju i spajaju
s zamrznutim panelima Faze 1 (benchmark + korelacijski hijerarhijski) u
jedinstveni prošireni panel prinosa.

In [3]:
factor_result = run_hierarchical_walk_forward(
    monthly_returns,
    factor_exposures,
    factor_clusters,
    correlation_clusters,
    metadata,
    windows,
    k=K,
    tree_space="factor",
    membership=membership,
)
factor_returns = factor_result["port_returns_panel"]
factor_weights = factor_result["weights_panel"]
factor_status = factor_result["portfolio_status"]

factor_returns.to_csv(TABLES_DIR / "11_port_returns_panel_factor.csv")
factor_weights.to_csv(TABLES_DIR / "11_weights_panel_factor.csv", index=False)
factor_status.to_csv(TABLES_DIR / "11_portfolio_status_factor.csv", index=False)

# Spoj sa zamrznutim panelima Faze 1 u prošireni panel prinosa.
benchmark_panel = pd.read_csv(
    TABLES_DIR / "05_port_returns_panel_extended.csv", index_col=0, parse_dates=True
)
corr_hier_panel = pd.read_csv(
    TABLES_DIR / "09_port_returns_panel_hierarchical.csv", index_col=0, parse_dates=True
)
extended_panel = pd.concat([benchmark_panel, corr_hier_panel, factor_returns], axis=1)

print("Faktorski panel:", factor_returns.shape, "| stupci:", list(factor_returns.columns))
print("Prošireni panel:", extended_panel.shape)
print("\nUdio težine na capu (capped_weight_share) — faktorski prostor:")
print(factor_status.groupby("portfolio")["capped_weight_share"].agg(["mean", "max"]).round(4))

Faktorski panel: (156, 3) | stupci: ['hrp_factor', 'herc_factor', 'nco_factor']
Prošireni panel: (156, 13)

Udio težine na capu (capped_weight_share) — faktorski prostor:
               mean   max
portfolio                
herc_factor  0.0038  0.05
hrp_factor   0.0000  0.00
nco_factor   0.1308  0.45


In [4]:
# Kriterij prihvaćanja F3.2: 3 stupca prinosa, čist status, dovoljno mjeseci.
assert list(factor_returns.columns) == list(HIERARCHICAL_FACTOR_PORTFOLIO_NAMES)
assert "capped_weight_share" in factor_status.columns
failed = factor_status.loc[factor_status["status"].str.startswith("failed")]
assert failed.empty, f"Neobjašnjeni neuspjesi:\n{failed}"

# Zamrznuti raspon 2013–2025 (korekcija 2026-06-13: backtest_start=2008-01 →
# 13 prozora × 12 = 156 testnih mjeseci po alokatoru). Izvorni prag „≥240” iz
# TASKS.md odnosio se na predkorekcijski raspon 2005–2025; pratimo isti broj
# mjeseci kao zamrznuti panel Faze 1 (09_port_returns_panel_hierarchical.csv).
n_months = factor_returns.notna().sum()
assert (n_months == len(corr_hier_panel)).all()
print("Statusi:", factor_status["status"].unique().tolist())
print("n_months po alokatoru:", n_months.to_dict())

Statusi: ['ok']
n_months po alokatoru: {'hrp_factor': 156, 'herc_factor': 156, 'nco_factor': 156}


## 11.2 F3.3 — Parne razlike korelacijska↔faktorska (test H2, dio nagiba)

Kontrolirane parne usporedbe `factor − corr` po alokatoru, **zajedničkim**
blok-bootstrapom (`block_bootstrap_diff`, sjeme/broj uzoraka/blok iz
`config.yaml`). Primarni ishod je **koncentracija stila**; sekundarni su
ostvarena volatilnost (test ne-inferiornosti), Sharpe (bruto i neto na 10 bps)
i maksimalni drawdown. Sharpe se bootstrapira na nizovima viška iznad `RF`
(neto = nakon troška na refit obrtaj), pa zajednički ponovno uzorkovani
mjeseci ostaju poravnani.

**Fusnota (obavezno, F3.3) — isključenje `hrp_corr_single`:** za HRP se
uspoređuje `hrp_factor` s `hrp_corr_ward` (Wardova veza u oba kraka, korekcija
K1). `hrp_corr_single` (jednostruka veza, vjerna replikacija López de Prada
2016) **isključen je iz uzročne usporedbe** jer bi istovremeno mijenjao metodu
veze i prostor (konfundiranje); ostaje isključivo replikacijski red Faze 1.
HERC i NCO koriste Wardovu vezu u oba prostora po konstrukciji.

**Pravilo odluke (H2, dio nagiba):** faktorski prostor *NE* neutralizira
stilski nagib ako 95 % CI razlike koncentracije stila (`factor − corr_ward`)
obuhvaća 0. Ne-inferiornost volatilnosti: gornja granica 95 % CI razlike
ostvarene volatilnosti ≤ **+1 p.b. (+0.01)**.

In [5]:
def _sc(series, fac):
    return style_concentration(series, fac)


def _vol(series):
    return annualized_vol(series)


def _mdd(series):
    return max_drawdown(series)


def _sharpe0(series):
    # niz je već višak iznad RF → rf=0 ostaje konzistentan pod ponovnim uzorkovanjem
    return sharpe_ratio(series, 0.0)


# Neto paneli (trošak na refit obrtaj) za obje strane usporedbe.
weights_corr = pd.read_csv(TABLES_DIR / "09_weights_panel_hierarchical.csv")
weights_all = pd.concat([weights_corr, factor_weights], ignore_index=True)
turnover = turnover_per_window(weights_all)

gross = pd.concat([corr_hier_panel, factor_returns], axis=1)
net = apply_costs(gross, turnover, tc_bps=TC_BPS)
rf = factors["RF"].reindex(gross.index).fillna(0.0)
gross_excess = gross.sub(rf, axis=0)
net_excess = net.sub(rf, axis=0)

# (oznaka alokatora, faktorski stupac, korelacijska baza s Wardovom vezom — K1)
pairs = [
    ("hrp", "hrp_factor", "hrp_corr_ward"),
    ("herc", "herc_factor", "herc_corr"),
    ("nco", "nco_factor", "nco_corr"),
]

BOOT = dict(n_bootstraps=N_BOOTSTRAP_RETURNS, block_size=BLOCK_SIZE_MONTHS, seed=RANDOM_SEED)
diff_rows = []
for allocator, fac_name, corr_name in pairs:
    n_common = int(
        pd.concat([gross[fac_name], gross[corr_name]], axis=1).dropna().shape[0]
    )
    results = {
        "style_concentration": block_bootstrap_diff(
            gross[fac_name], gross[corr_name], metric_fn=_sc, factors=factors, **BOOT
        ),
        "annualized_vol": block_bootstrap_diff(
            gross[fac_name], gross[corr_name], metric_fn=_vol, **BOOT
        ),
        "sharpe_gross": block_bootstrap_diff(
            gross_excess[fac_name], gross_excess[corr_name], metric_fn=_sharpe0, **BOOT
        ),
        "sharpe_net": block_bootstrap_diff(
            net_excess[fac_name], net_excess[corr_name], metric_fn=_sharpe0, **BOOT
        ),
        "max_drawdown": block_bootstrap_diff(
            gross[fac_name], gross[corr_name], metric_fn=_mdd, **BOOT
        ),
    }
    for metric_name, r in results.items():
        diff_rows.append(
            {
                "allocator": allocator,
                "metric": metric_name,
                "point": r["point"],
                "ci_low": r["ci_low"],
                "ci_high": r["ci_high"],
                "p_two_sided": r["p_two_sided"],
                "n_months": n_common,
            }
        )

paired_diffs_space = pd.DataFrame(
    diff_rows,
    columns=["allocator", "metric", "point", "ci_low", "ci_high", "p_two_sided", "n_months"],
)
paired_diffs_space.to_csv(TABLES_DIR / "11_paired_diffs_space.csv", index=False)
print(paired_diffs_space.round(4).to_string(index=False))

allocator              metric   point  ci_low  ci_high  p_two_sided  n_months
      hrp style_concentration  0.0118 -0.0155   0.0460        0.426       156
      hrp      annualized_vol  0.0003 -0.0006   0.0013        0.542       156
      hrp        sharpe_gross  0.0053 -0.0184   0.0305        0.714       156
      hrp          sharpe_net  0.0054 -0.0184   0.0306        0.712       156
      hrp        max_drawdown  0.0060 -0.0089   0.0107        0.784       156
     herc style_concentration  0.0882 -0.5393   0.2428        0.914       156
     herc      annualized_vol  0.0093 -0.0039   0.0219        0.172       156
     herc        sharpe_gross -0.0454 -0.3298   0.2497        0.776       156
     herc          sharpe_net -0.0452 -0.3311   0.2500        0.778       156
     herc        max_drawdown -0.0134 -0.0810   0.0422        0.474       156
      nco style_concentration -0.0613 -0.1538   0.1057        0.528       156
      nco      annualized_vol  0.0012 -0.0039   0.0065        0.

In [6]:
# Kriterij F3.3: 3 alokatora × ≥4 metrike; HRP redak uspoređuje hrp_factor s
# hrp_corr_ward (vidi fusnotu gore).
assert set(paired_diffs_space["allocator"]) == {"hrp", "herc", "nco"}
assert paired_diffs_space.groupby("allocator")["metric"].nunique().min() >= 4

sc = paired_diffs_space.query("metric == 'style_concentration'").set_index("allocator")
vol = paired_diffs_space.query("metric == 'annualized_vol'").set_index("allocator")
print("H2 (dio nagiba) — faktorski prostor NE neutralizira nagib ako CI Δstyle obuhvaća 0:")
for a in ["hrp", "herc", "nco"]:
    row = sc.loc[a]
    covers0 = bool(row["ci_low"] <= 0 <= row["ci_high"])
    print(
        f"  {a:>4}: Δstyle = {row['point']:+.3f} "
        f"[{row['ci_low']:+.3f}, {row['ci_high']:+.3f}], p={row['p_two_sided']:.3f}, "
        f"CI obuhvaća 0: {covers0}"
    )
print("\nNe-inferiornost volatilnosti (gornja granica 95% CI ≤ +0.01 = +1 p.b.):")
for a in ["hrp", "herc", "nco"]:
    row = vol.loc[a]
    noninf = bool(row["ci_high"] <= 0.01)
    print(
        f"  {a:>4}: Δvol = {row['point']:+.4f} "
        f"[{row['ci_low']:+.4f}, {row['ci_high']:+.4f}], ne-inferiorno: {noninf}"
    )

H2 (dio nagiba) — faktorski prostor NE neutralizira nagib ako CI Δstyle obuhvaća 0:
   hrp: Δstyle = +0.012 [-0.016, +0.046], p=0.426, CI obuhvaća 0: True
  herc: Δstyle = +0.088 [-0.539, +0.243], p=0.914, CI obuhvaća 0: True
   nco: Δstyle = -0.061 [-0.154, +0.106], p=0.528, CI obuhvaća 0: True

Ne-inferiornost volatilnosti (gornja granica 95% CI ≤ +0.01 = +1 p.b.):
   hrp: Δvol = +0.0003 [-0.0006, +0.0013], ne-inferiorno: True
  herc: Δvol = +0.0093 [-0.0039, +0.0219], ne-inferiorno: False
   nco: Δvol = +0.0012 [-0.0039, +0.0065], ne-inferiorno: True


## 11.3 F3.4 — Stabilnost klastera (ARI) i obrtaj (test H2, dio 2)

Hipoteza H2 tvrdi da faktorski prostor, iako ne neutralizira nagib (F3.3),
donosi dobitke u **stabilnosti klastera**, **obrtaju** i **interpretabilnosti**.

Stabla se za obje strane režu na **isti K** uz **Wardovu vezu u oba kraka**
(K1): korelacijsko stablo = ward varijanta iz F1.1b (`build_correlation_tree`),
faktorsko = `factor_cluster` na standardiziranim FF5 značajkama. Oznake se
računaju na **istom backtest univerzumu** koji alokatori vide po prozoru.
`ari_between_consecutive_windows` daje ARI na presjeku dionica za svaki
prijelaz; ARI je invarijantan na permutaciju oznaka pa je usporediv između
prostora.

Napomena o broju redaka: zamrznuti raspon je 13 prozora (2013–2025; korekcija
2026-06-13), pa ARI tablica ima **12 prijelaza po prostoru** (izvorni TASKS-ov
broj „20 / 21 prozor” odnosio se na predkorekcijski raspon 2005–2025).

In [7]:
# Backtest univerzum po prozoru (identičan u oba prostora; iz faktorskog panela težina).
universe_by_window = factor_weights.groupby("train_window")["ticker"].apply(
    lambda s: sorted(set(s))
)

# Oznake klastera kakve alokatori stvarno vide: korelacijsko WARD stablo (F1.1b)
# i faktorsko stablo, oba rezana na K, na istom univerzumu po prozoru.
corr_label_rows, factor_label_rows = [], []
for window in windows:
    label = window.label
    universe = universe_by_window.loc[label]
    train_panel = (
        monthly_returns.loc[window.train_start:window.train_end, universe]
        .dropna(axis=0, how="any")
    )
    cols = list(train_panel.columns)
    ward = build_correlation_tree(train_panel.corr(), linkage="ward")
    corr_labels = fcluster(ward, t=K, criterion="maxclust")
    corr_label_rows += [
        {"train_window": label, "ticker": t, "cluster": int(c)}
        for t, c in zip(cols, corr_labels)
    ]
    feats = (
        factor_exposures.loc[factor_exposures["train_window"] == label]
        .set_index("ticker")
        .loc[cols, FACTOR_FEATURE_COLUMNS]
    )
    factor_labels, _ = factor_cluster(feats, K)
    factor_label_rows += [
        {"train_window": label, "ticker": t, "cluster": int(c)}
        for t, c in factor_labels.items()
    ]

corr_clusters_bt = pd.DataFrame(corr_label_rows)
factor_clusters_bt = pd.DataFrame(factor_label_rows)

ari_corr = ari_between_consecutive_windows(corr_clusters_bt, "cluster").assign(space="correlation")
ari_factor = ari_between_consecutive_windows(factor_clusters_bt, "cluster").assign(space="factor")
ari_stability = pd.concat([ari_corr, ari_factor], ignore_index=True)[
    ["space", "from_window", "to_window", "n_common", "ari"]
]
ari_stability.to_csv(TABLES_DIR / "11_ari_stability.csv", index=False)

print("Prijelaza po prostoru:", ari_corr.shape[0], "(korelacijska),", ari_factor.shape[0], "(faktorska)")
print("\nARI (prosjek ± sd po prostoru — viši = stabilnije):")
print(ari_stability.groupby("space")["ari"].agg(["mean", "std", "count"]).round(4))

Prijelaza po prostoru: 12 (korelacijska), 12 (faktorska)

ARI (prosjek ± sd po prostoru — viši = stabilnije):
               mean     std  count
space                             
correlation  0.2935  0.0509     12
factor       0.2513  0.0448     12


In [8]:
# Obrtaj korelacijskih naspram faktorskih verzija po alokatoru (HRP = hrp_corr_ward).
space_to_portfolio = {
    ("hrp", "correlation"): "hrp_corr_ward",
    ("hrp", "factor"): "hrp_factor",
    ("herc", "correlation"): "herc_corr",
    ("herc", "factor"): "herc_factor",
    ("nco", "correlation"): "nco_corr",
    ("nco", "factor"): "nco_factor",
}
turnover_rows = []
for (allocator, space), portfolio in space_to_portfolio.items():
    sub = turnover.loc[turnover["portfolio"] == portfolio, "turnover"]
    turnover_rows.append(
        {
            "allocator": allocator,
            "space": space,
            "mean_turnover": float(sub.mean()),
            "n_windows": int(sub.shape[0]),
        }
    )
turnover_by_space = (
    pd.DataFrame(turnover_rows)
    .sort_values(["allocator", "space"])
    .reset_index(drop=True)
)
turnover_by_space.to_csv(TABLES_DIR / "11_turnover_by_space.csv", index=False)
print(turnover_by_space.round(4).to_string(index=False))

allocator       space  mean_turnover  n_windows
     herc correlation         0.6006         13
     herc      factor         0.5875         13
      hrp correlation         0.2542         13
      hrp      factor         0.2456         13
      nco correlation         0.5643         13
      nco      factor         0.6049         13


In [9]:
# Argument redukcije dimenzije: korelacijski prostor procjenjuje ~N(N-1)/2
# korelacija po prozoru, faktorski 6N beta/karakteristika (FACTOR_FEATURE_COLUMNS).
dim_rows = []
for window in windows:
    n = len(universe_by_window.loc[window.label])
    n_corr = n * (n - 1) // 2
    n_factor = len(FACTOR_FEATURE_COLUMNS) * n
    dim_rows.append(
        {
            "train_window": window.label,
            "n_assets": n,
            "corr_estimates": n_corr,
            "factor_estimates": n_factor,
            "ratio_corr_to_factor": n_corr / n_factor,
        }
    )
dim_reduction = pd.DataFrame(dim_rows)
print(dim_reduction.to_string(index=False))
print(
    f"\nProsječno: korelacijski prostor procjenjuje "
    f"{dim_reduction['ratio_corr_to_factor'].mean():.1f}× više veličina nego faktorski "
    f"(~N(N-1)/2 korelacija naspram {len(FACTOR_FEATURE_COLUMNS)}N karakteristika)."
)

train_window  n_assets  corr_estimates  factor_estimates  ratio_corr_to_factor
     2012-12       347           60031              2082             28.833333
     2013-12       348           60378              2088             28.916667
     2014-12       356           63190              2136             29.583333
     2015-12       366           66795              2196             30.416667
     2016-12       389           75466              2334             32.333333
     2017-12       397           78606              2382             33.000000
     2018-12       408           83028              2448             33.916667
     2019-12       424           89676              2544             35.250000
     2020-12       437           95266              2622             36.333333
     2021-12       451          101475              2706             37.500000
     2022-12       466          108345              2796             38.750000
     2023-12       472          111156              

## 11.4 F3.5 — Odluka o K: primarno + robusnost

**Primarna analiza** drži isti K za obje verzije: `primary_k` iz `config.yaml`
(domenski izbor K=10; HRP je K-neovisan, K ulazi samo u HERC/NCO — zaključana
odluka 5/8). **Robusnost** ponavlja HERC/NCO s vlastiti-optimalnim K svake
verzije: korelacijska = silueta na **korelacijskoj udaljenosti** (Wardovo
stablo), faktorska = postojeći prelet siluete na FF5 značajkama
(`02_silhouette_by_k.csv`). Dodatno: **k-means particija za NCO** (riješeno
pitanje 7) — alternativa rezu stabla, na standardiziranim FF5 značajkama
(faktorski) odn. korelacijskoj udaljenosti (korelacijski).

Izvještavaju se ključne metrike (koncentracija stila, ostvarena vol, obrtaj)
za sve kombinacije {K-varijanta} × {HERC, NCO} × {prostor}, radi osjetljivosti.

In [10]:
# Vlastiti-optimalni K po prostoru.
corr_sil_records = []
for k in range(2, 13):
    sils = []
    for window in windows:
        universe = universe_by_window.loc[window.label]
        train_panel = (
            monthly_returns.loc[window.train_start:window.train_end, universe]
            .dropna(axis=0, how="any")
        )
        corr = train_panel.corr().clip(lower=-1.0, upper=1.0)
        dist = np.sqrt(2.0 * (1.0 - corr)).to_numpy().copy()
        np.fill_diagonal(dist, 0.0)
        labels = fcluster(build_correlation_tree(corr, "ward"), t=k, criterion="maxclust")
        if 1 < len(set(labels)) < len(labels):
            sils.append(silhouette_score(dist, labels, metric="precomputed"))
    corr_sil_records.append({"k": k, "mean_silhouette": float(np.mean(sils))})
corr_silhouette = pd.DataFrame(corr_sil_records)
k_own_corr = int(corr_silhouette.loc[corr_silhouette["mean_silhouette"].idxmax(), "k"])

factor_silhouette = pd.read_csv(TABLES_DIR / "02_silhouette_by_k.csv")
k_own_factor = int(factor_silhouette.loc[factor_silhouette["mean_silhouette"].idxmax(), "k"])
k_primary = int(PRIMARY_K)
print(f"K_primary = {k_primary} | K_own_corr = {k_own_corr} | K_own_factor = {k_own_factor}")

K_primary = 10 | K_own_corr = 2 | K_own_factor = 2


In [11]:
def _allocator_metrics(space, k):
    """Metrike (style/vol/turnover) za HERC i NCO u danom prostoru i K."""
    res = run_hierarchical_walk_forward(
        monthly_returns, factor_exposures, factor_clusters, correlation_clusters,
        metadata, windows, k=k, tree_space=space, membership=membership,
    )
    panel = res["port_returns_panel"]
    wt = turnover_per_window(res["weights_panel"])
    suffix = "corr" if space == "correlation" else "factor"
    out = {}
    for allocator in ("herc", "nco"):
        name = f"{allocator}_{suffix}"
        series = panel[name]
        out[allocator] = {
            "style_concentration": style_concentration(series, factors),
            "ann_vol": annualized_vol(series),
            "turnover": float(wt.loc[wt["portfolio"] == name, "turnover"].mean()),
        }
    return out


def _kmeans_nco_metrics(space, k, seed=RANDOM_SEED):
    """NCO s k-means particijom (umjesto reza stabla) — riješeno pitanje 7."""
    pieces, weight_rows = [], []
    for window in windows:
        universe = universe_by_window.loc[window.label]
        train_panel = (
            monthly_returns.loc[window.train_start:window.train_end, universe]
            .dropna(axis=0, how="any")
        )
        cols = list(train_panel.columns)
        sigma = ledoit_wolf_cov(train_panel).loc[cols, cols]
        if space == "factor":
            feats = (
                factor_exposures.loc[factor_exposures["train_window"] == window.label]
                .set_index("ticker")
                .loc[cols, FACTOR_FEATURE_COLUMNS]
            )
            features = StandardScaler().fit_transform(feats.to_numpy(dtype=float))
        else:
            corr = train_panel.corr().clip(lower=-1.0, upper=1.0)
            features = np.sqrt(2.0 * (1.0 - corr)).to_numpy()
        labels = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(features)
        weights, _ = nco_weights(sigma, pd.Series(labels, index=cols), w_max=W_MAX)
        weights = weights.reindex(cols).fillna(0.0)
        weight_rows += [
            {"train_window": window.label, "portfolio": f"nco_kmeans_{space}",
             "ticker": t, "weight": float(v)}
            for t, v in weights.items()
        ]
        test = monthly_returns.loc[window.test_start:window.test_end, cols]
        pieces.append(test.loc[:, weights.index].fillna(0.0).dot(weights))
    series = pd.concat(pieces).sort_index()
    wt = turnover_per_window(pd.DataFrame(weight_rows))
    return {
        "style_concentration": style_concentration(series, factors),
        "ann_vol": annualized_vol(series),
        "turnover": float(wt["turnover"].mean()),
    }


k_variants = [("primary", k_primary), ("own_corr", k_own_corr), ("own_factor", k_own_factor)]
robustness_rows = []
cache = {}
for space in ("correlation", "factor"):
    for variant, k in k_variants:
        if (space, k) not in cache:
            cache[(space, k)] = _allocator_metrics(space, k)
        metrics = cache[(space, k)]
        for allocator in ("herc", "nco"):
            robustness_rows.append(
                {"allocator": allocator, "space": space, "k_variant": variant, "k": k,
                 **metrics[allocator]}
            )
# Dodatna robusnost: k-means particija za NCO (oba prostora, primarni K).
for space in ("correlation", "factor"):
    m = _kmeans_nco_metrics(space, k_primary)
    robustness_rows.append(
        {"allocator": "nco", "space": space, "k_variant": "kmeans", "k": k_primary, **m}
    )

k_robustness = pd.DataFrame(
    robustness_rows,
    columns=["allocator", "space", "k_variant", "k",
             "style_concentration", "ann_vol", "turnover"],
)
k_robustness.to_csv(TABLES_DIR / "11_k_robustness.csv", index=False)
print(k_robustness.round(4).to_string(index=False))

allocator       space  k_variant  k  style_concentration  ann_vol  turnover
     herc correlation    primary 10               0.5739   0.1431    0.6006
      nco correlation    primary 10               0.7982   0.1234    0.5643
     herc correlation   own_corr  2               0.5579   0.1392    0.2917
      nco correlation   own_corr  2               0.7943   0.1291    0.5534
     herc correlation own_factor  2               0.5579   0.1392    0.2917
      nco correlation own_factor  2               0.7943   0.1291    0.5534
     herc      factor    primary 10               0.6621   0.1525    0.5875
      nco      factor    primary 10               0.7370   0.1247    0.6049
     herc      factor   own_corr  2               0.5959   0.1488    0.2036
      nco      factor   own_corr  2               0.8801   0.1279    0.5557
     herc      factor own_factor  2               0.5959   0.1488    0.2036
      nco      factor own_factor  2               0.8801   0.1279    0.5557
      nco co

In [12]:
# Kriterij F3.5: metrike za {primarni K, vlastiti K korelacijski, vlastiti K
# faktorski, kmeans-NCO} × {HERC, NCO}.
assert set(k_robustness["k_variant"]) >= {"primary", "own_corr", "own_factor", "kmeans"}
assert set(k_robustness["allocator"]) == {"herc", "nco"}
assert {"style_concentration", "ann_vol", "turnover"}.issubset(k_robustness.columns)

print("Osjetljivost koncentracije stila na K (primarni naspram vlastiti-optimalni):")
pivot = k_robustness.pivot_table(
    index=["allocator", "space"], columns="k_variant", values="style_concentration"
)
print(pivot.round(3))

Osjetljivost koncentracije stila na K (primarni naspram vlastiti-optimalni):
k_variant              kmeans  own_corr  own_factor  primary
allocator space                                             
herc      correlation     NaN     0.558       0.558    0.574
          factor          NaN     0.596       0.596    0.662
nco       correlation   0.795     0.794       0.794    0.798
          factor        0.765     0.880       0.880    0.737
